In [ ]:
from dotenv import load_dotenv
import numpy as np
from numpy import dot
from numpy.linalg import norm
import pandas as pd
from langchain_openai import OpenAIEmbeddings

load_dotenv("../../.env")

True

In [2]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
query_result = embeddings.embed_query('설악산의 가을은 아름답습니다.')
print(query_result)

[0.05120849609375, 0.01087188720703125, -0.008514404296875, 0.034912109375, 0.029876708984375, -0.0036602020263671875, -0.0168609619140625, -0.0022125244140625, -0.004917144775390625, -0.0361328125, -0.04351806640625, 0.011993408203125, 0.05352783203125, 0.0318603515625, -0.01418304443359375, -0.0005068778991699219, 0.00875091552734375, 0.01305389404296875, -0.0032482147216796875, 0.007556915283203125, -0.00809478759765625, -0.01174163818359375, 0.0200347900390625, 0.01296234130859375, -0.03204345703125, 0.02978515625, 0.035003662109375, 0.031951904296875, -0.01509857177734375, 0.0122528076171875, 0.031890869140625, 0.00539398193359375, 0.02032470703125, 0.05908203125, -0.024017333984375, 0.054779052734375, 0.006786346435546875, 0.050933837890625, 0.045135498046875, 0.0257415771484375, -0.0247650146484375, 0.00995635986328125, 0.04296875, 0.0143890380859375, -0.0103607177734375, -0.038787841796875, 0.0199432373046875, -0.0192108154296875, 0.0188140869140625, 0.022705078125, -0.00177764

In [5]:
print(len(query_result))

3072


- 한국어의 단어 수는 51만 ~ 120만

In [6]:
data = [
    '안전성은 원금 손실을 가져오지 않을 가능성을 말합니다',
    '수익성은 금융상품의 가격 상승이나 이자 수익을 기대할 수 있는 정도를 말합니다',
    '유동성은 쉽게 현금으로 전환할 수 있는 정도로서, 환금성이라고도 합니다',
    '2040 서울도시기본계획은 20년 후 서울의 모습을 내다보고 시대상 및 가치를 담은 미래상과 발전 방향의 기본골격을 제시하는 장기계획이다',
    '도시기본계획은 복지·경제·환경·교통·안전 등 다양한 부문을 포괄하는 종합계획으로 서울시 정책의 전 분야를 균형 있게 통합·조정하는 공간분야 최상위 계획의 위상을 갖는다',
    '도시기본계획은 다른 법률에 따른 토지이용·개발 및 보전계획의 기본이 되며, 부문별 법정계획과 시정 운영계획 등 서울시 모든 계획 수립 시 방향성을 제시하는 역할을 수행한다',
]
df = pd.DataFrame(data, columns=['text'])
df

,text
0,안전성은 원금 손실을 가져오지 않을 가능성을 말합니다
1,수익성은 금융상품의 가격 상승이나 이자 수익을 기대할 수 있는 정도를 말합니다
2,"유동성은 쉽게 현금으로 전환할 수 있는 정도로서, 환금성이라고도 합니다"
3,2040 서울도시기본계획은 20년 후 서울의 모습을 내다보고 시대상 및 가치를 담은...
4,도시기본계획은 복지·경제·환경·교통·안전 등 다양한 부문을 포괄하는 종합계획으로 서...
5,"도시기본계획은 다른 법률에 따른 토지이용·개발 및 보전계획의 기본이 되며, 부문별 ..."


In [7]:
# 텍스트를 임베딩 벡터로 변환하는 함수 정의
def text_to_vector(text):
  return embeddings.embed_query(text)

# DataFrame의 각 행에 대해 'text' 열의 내용을 임베딩 벡터로 변환
df['embedding'] = df.apply(lambda row: text_to_vector(row.text), axis=1)

# 변환된 DataFrame 출력
df

,text,embedding
0,안전성은 원금 손실을 가져오지 않을 가능성을 말합니다,"[0.006599, 0.003338, -0.006763, 0.024719, 0.01..."
1,수익성은 금융상품의 가격 상승이나 이자 수익을 기대할 수 있는 정도를 말합니다,"[-0.016129, -0.007713, -0.015808, -0.017914, 0..."
2,"유동성은 쉽게 현금으로 전환할 수 있는 정도로서, 환금성이라고도 합니다","[-0.005596, -0.000361, -0.007874, 0.003563, -0..."
3,2040 서울도시기본계획은 20년 후 서울의 모습을 내다보고 시대상 및 가치를 담은...,"[0.019577, -0.035706, -0.019257, 0.014145, 0.0..."
4,도시기본계획은 복지·경제·환경·교통·안전 등 다양한 부문을 포괄하는 종합계획으로 서...,"[0.036926, -0.032959, -0.007164, 0.003656, 0.0..."
5,"도시기본계획은 다른 법률에 따른 토지이용·개발 및 보전계획의 기본이 되며, 부문별 ...","[0.053284, -0.061127, -0.011101, 0.007229, 0.0..."


In [ ]:
# 코사인 유사도 계산 함수
def cosine_similarity(vec1, vec2):
    """두 벡터 간의 코사인 유사도를 계산합니다."""
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2 + 1e-9)  # 작은 값 추가로 0 나누기 방지

# 주어진 쿼리와 가장 유사한 상위 3개의 문서를 반환하는 함수
def get_answer_candidate(df, query):
    # 쿼리 텍스트를 임베딩 벡터로 변환
    query_embedding = text_to_vector(query)

    # DataFrame의 각 문서 임베딩과 쿼리 임베딩 간의 유사도 계산
    df["similarity"] = df.embedding.apply(lambda x: cosine_similarity(np.array(x), np.array(query_embedding)))

    # 유사도가 높은 순으로 정렬하고 상위 3개 문서 선택
    top_three_doc = df.sort_values("similarity", ascending=False).head(3)

    return top_three_doc

# 예시 쿼리로 유사한 문서 검색
sim_result = get_answer_candidate(df, '금융 상품')
sim_result

,text,embedding,similarity
1,수익성은 금융상품의 가격 상승이나 이자 수익을 기대할 수 있는 정도를 말합니다,"[-0.016129, -0.007713, -0.015808, -0.017914, 0...",0.514120
2,"유동성은 쉽게 현금으로 전환할 수 있는 정도로서, 환금성이라고도 합니다","[-0.005596, -0.000361, -0.007874, 0.003563, -0...",0.348765
0,안전성은 원금 손실을 가져오지 않을 가능성을 말합니다,"[0.006599, 0.003338, -0.006763, 0.024719, 0.01...",0.298347
